# FinGuard AI - Risk Scoring & Analysis

## Scenario: FraudShield Risk Assessment

**Context:** FinGuard AI operates FraudShield, a GenAI-powered fraud detection system for retail banking across the EU. Recent incidents include socioeconomic bias (lower-income zip codes flagged disproportionately) and adversarial evasion techniques.

**Objective:** Calculate risk scores, rank risks, analyze distributions, and generate visualizations and an executive summary report.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## Load Risk Data

In [ ]:
# Define all 15 risks from the completed Risk Register
risks = [
    {
        'id': 'R001',
        'nist_function': 'Measure',
        'category': 'Technical',
        'description': 'Model drift: FraudShield performance degrades as transaction patterns evolve',
        'impact': 4,
        'likelihood': 4,
    },
    {
        'id': 'R002',
        'nist_function': 'Govern',
        'category': 'Ethical',
        'description': 'Socioeconomic bias: Lower-income zip codes flagged disproportionately',
        'impact': 5,
        'likelihood': 4,
    },
    {
        'id': 'R003',
        'nist_function': 'Manage',
        'category': 'Technical',
        'description': 'Adversarial evasion: Fraudsters bypass detection patterns',
        'impact': 5,
        'likelihood': 3,
    },
    {
        'id': 'R004',
        'nist_function': 'Measure',
        'category': 'Technical',
        'description': 'Data pipeline failures: Missing or corrupted transaction feeds',
        'impact': 4,
        'likelihood': 3,
    },
    {
        'id': 'R005',
        'nist_function': 'Manage',
        'category': 'Technical',
        'description': 'False positive flood: Legitimate transactions blocked at high rates',
        'impact': 3,
        'likelihood': 4,
    },
    {
        'id': 'R006',
        'nist_function': 'Govern',
        'category': 'Ethical',
        'description': 'Lack of explainability: Customers receive no explanation for blocking',
        'impact': 3,
        'likelihood': 4,
    },
    {
        'id': 'R007',
        'nist_function': 'Govern',
        'category': 'Legal',
        'description': 'GDPR automated decision violation: Blocking without human review (Article 22)',
        'impact': 5,
        'likelihood': 2,
    },
    {
        'id': 'R008',
        'nist_function': 'Map',
        'category': 'Legal',
        'description': 'Right to explanation: Customers not informed per GDPR Articles 13-14',
        'impact': 4,
        'likelihood': 3,
    },
    {
        'id': 'R009',
        'nist_function': 'Manage',
        'category': 'Operational',
        'description': 'System downtime: Unavailability during peak hours leaves transactions unmonitored',
        'impact': 4,
        'likelihood': 2,
    },
    {
        'id': 'R010',
        'nist_function': 'Map',
        'category': 'Operational',
        'description': 'Staff training gaps: Analysts lack understanding of model limitations',
        'impact': 3,
        'likelihood': 3,
    },
    {
        'id': 'R011',
        'nist_function': 'Manage',
        'category': 'Operational',
        'description': 'Third-party vendor dependency: Reliance on external feeds with no backup',
        'impact': 3,
        'likelihood': 2,
    },
    {
        'id': 'R012',
        'nist_function': 'Measure',
        'category': 'Technical',
        'description': 'Concept drift: New fraud patterns emerge not in training data',
        'impact': 4,
        'likelihood': 4,
    },
    {
        'id': 'R013',
        'nist_function': 'Map',
        'category': 'Ethical',
        'description': 'Unequal error rates: False positives differ across demographic segments',
        'impact': 3,
        'likelihood': 4,
    },
    {
        'id': 'R014',
        'nist_function': 'Govern',
        'category': 'Legal',
        'description': 'Data retention non-compliance: Transaction data stored beyond limits',
        'impact': 3,
        'likelihood': 2,
    },
    {
        'id': 'R015',
        'nist_function': 'Measure',
        'category': 'Technical',
        'description': 'Model interpretability gap: Cannot audit specific fraud decisions',
        'impact': 3,
        'likelihood': 3,
    },
]

# Create DataFrame
df = pd.DataFrame(risks)
print(f'Loaded {len(df)} risks')
print(f'Categories: {df["category"].nunique()}')
print(f'NIST Functions: {df["nist_function"].nunique()}')

## Calculate Risk Scores

In [ ]:
# Calculate risk_score = Impact * Likelihood
df['risk_score'] = df['impact'] * df['likelihood']

# Assign risk_level based on thresholds
def assign_risk_level(score):
    if score >= 15:
        return 'Critical'
    elif score >= 10:
        return 'High'
    elif score >= 5:
        return 'Medium'
    else:
        return 'Low'

df['risk_level'] = df['risk_score'].apply(assign_risk_level)

# Display the DataFrame with calculated scores
print('Risk Scores Calculated:')
print(df[['id', 'category', 'impact', 'likelihood', 'risk_score', 'risk_level']].to_string())

## Ranked Risk Register

In [ ]:
# Sort risks by risk_score (descending)
df_ranked = df.sort_values('risk_score', ascending=False).reset_index(drop=True)
df_ranked['rank'] = range(1, len(df_ranked) + 1)

# Display top 10 ranked risks
print('\nTOP 10 RANKED RISKS:')
print(df_ranked[['rank', 'id', 'nist_function', 'category', 'description', 'risk_score', 'risk_level']].head(10).to_string())

## Risk Statistics

In [ ]:
print('\n' + '='*80)
print('RISK STATISTICS')
print('='*80)

# Count by risk level
print('\n--- RISK LEVEL DISTRIBUTION ---')
level_counts = df['risk_level'].value_counts().sort_index()
for level in ['Critical', 'High', 'Medium', 'Low']:
    count = level_counts.get(level, 0)
    pct = (count / len(df)) * 100
    print(f'{level:12s}: {count:2d} risks ({pct:5.1f}%)')

# Count by category
print('\n--- RISK BY CATEGORY ---')
cat_counts = df['category'].value_counts().sort_values(ascending=False)
for category, count in cat_counts.items():
    pct = (count / len(df)) * 100
    print(f'{category:15s}: {count:2d} risks ({pct:5.1f}%)')

# Count by NIST function
print('\n--- RISK BY NIST FUNCTION ---')
nist_counts = df['nist_function'].value_counts().sort_values(ascending=False)
for func, count in nist_counts.items():
    pct = (count / len(df)) * 100
    print(f'{func:12s}: {count:2d} risks ({pct:5.1f}%)')

# Score statistics
print('\n--- SCORE STATISTICS ---')
print(f'Average Risk Score: {df["risk_score"].mean():.2f}')
print(f'Median Risk Score:  {df["risk_score"].median():.2f}')
print(f'Max Risk Score:     {df["risk_score"].max()}')
print(f'Min Risk Score:     {df["risk_score"].min()}')

## Visualization 1: Risk Heat Map

In [ ]:
# Create risk heat map with background gradient and jittered points
fig, ax = plt.subplots(figsize=(12, 10))

# Create background heatmap showing risk zones
risk_matrix = np.zeros((5, 5))
for i in range(5):
    for j in range(5):
        risk_matrix[i, j] = (i + 1) * (j + 1)  # Impact * Likelihood

# Plot background gradient (risk zones)
im = ax.imshow(risk_matrix, cmap='RdYlGn_r', alpha=0.3, origin='lower', 
               extent=[0.5, 5.5, 0.5, 5.5], aspect='auto', vmin=1, vmax=25)

# Define distinct colors for categories
color_map = {
    'Technical': '#2563EB',   # Blue
    'Ethical': '#DC2626',     # Red
    'Legal': '#7C3AED',       # Purple
    'Operational': '#059669'  # Green
}

# Add small jitter to avoid overlapping points
np.random.seed(42)
jitter_strength = 0.15

# Plot each risk point with jitter
for idx, row in df.iterrows():
    jitter_x = np.random.uniform(-jitter_strength, jitter_strength)
    jitter_y = np.random.uniform(-jitter_strength, jitter_strength)
    x = row['impact'] + jitter_x
    y = row['likelihood'] + jitter_y
    
    ax.scatter(x, y, s=400, alpha=0.9, 
               color=color_map.get(row['category'], '#666666'),
               edgecolors='white', linewidth=2, zorder=5)
    
    # Add risk ID label with black text and white outline for readability
    ax.annotate(row['id'], (x, y), fontsize=8, ha='center', va='center', 
                fontweight='bold', color='white', zorder=6)

# Add quadrant divider lines
ax.axhline(y=3, color='#374151', linestyle='-', alpha=0.4, linewidth=2)
ax.axvline(x=3, color='#374151', linestyle='-', alpha=0.4, linewidth=2)

# Add quadrant labels with background boxes
bbox_props = dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='none')
ax.text(1.75, 4.5, 'MONITOR', fontsize=11, ha='center', fontweight='bold', 
        color='#B45309', bbox=bbox_props)
ax.text(4.25, 4.5, 'CRITICAL\nACTION', fontsize=11, ha='center', fontweight='bold', 
        color='#DC2626', bbox=bbox_props)
ax.text(1.75, 1.5, 'ACCEPT', fontsize=11, ha='center', fontweight='bold', 
        color='#059669', bbox=bbox_props)
ax.text(4.25, 1.5, 'MITIGATE', fontsize=11, ha='center', fontweight='bold', 
        color='#D97706', bbox=bbox_props)

# Configure axes
ax.set_xlabel('Impact (1=Low → 5=Critical)', fontsize=12, fontweight='bold')
ax.set_ylabel('Likelihood (1=Rare → 5=Almost Certain)', fontsize=12, fontweight='bold')
ax.set_title('FraudShield Risk Heat Map\nImpact vs. Likelihood by Category', fontsize=14, fontweight='bold')
ax.set_xlim(0.5, 5.5)
ax.set_ylim(0.5, 5.5)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_xticklabels(['1\nLow', '2\nMinor', '3\nModerate', '4\nMajor', '5\nCritical'])
ax.set_yticklabels(['1\nRare', '2\nUnlikely', '3\nPossible', '4\nLikely', '5\nAlmost\nCertain'])
ax.grid(True, alpha=0.3, color='gray', linestyle=':')

# Create legend with category markers
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, edgecolor='white', linewidth=1.5, label=category)
                   for category, color in color_map.items()]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10, title='Risk Category',
          title_fontsize=11, framealpha=0.9)

# Add colorbar for risk score reference
cbar = plt.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
cbar.set_label('Risk Score (Impact × Likelihood)', fontsize=10)
cbar.set_ticks([1, 5, 10, 15, 20, 25])

plt.tight_layout()
plt.show()

# Print legend for interpretation
print("\nHeat Map Interpretation:")
print("  • Upper-right (red zone): Critical risks requiring immediate action")
print("  • Lower-right (orange zone): High impact risks to mitigate")
print("  • Upper-left (yellow zone): Monitor closely for escalation")
print("  • Lower-left (green zone): Acceptable risks, periodic review")

## Visualization 2: Risk Scores by Category

In [ ]:
# Create grouped bar chart
fig, ax = plt.subplots(figsize=(12, 6))

category_stats = df.groupby('category')['risk_score'].agg(['mean', 'max', 'min']).round(2)
category_stats = category_stats.sort_values('max', ascending=False)

x = np.arange(len(category_stats))
width = 0.25

bars1 = ax.bar(x - width, category_stats['min'], width, label='Minimum', color='#70AD47', alpha=0.8)
bars2 = ax.bar(x, category_stats['mean'], width, label='Average', color='#4472C4', alpha=0.8)
bars3 = ax.bar(x + width, category_stats['max'], width, label='Maximum', color='#FF6B6B', alpha=0.8)

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Risk Category', fontsize=12, fontweight='bold')
ax.set_ylabel('Risk Score', fontsize=12, fontweight='bold')
ax.set_title('Risk Score Statistics by Category', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(category_stats.index)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Visualization 3: Risk Level Distribution

In [ ]:
# Create pie chart
fig, ax = plt.subplots(figsize=(10, 8))

level_dist = df['risk_level'].value_counts()
level_order = ['Critical', 'High', 'Medium', 'Low']
level_dist = level_dist.reindex([level for level in level_order if level in level_dist.index])

color_map_levels = {
    'Critical': '#F8696B',
    'High': '#FFD966',
    'Medium': '#C6EFCE',
    'Low': '#70AD47'
}
colors = [color_map_levels.get(level, 'gray') for level in level_dist.index]

wedges, texts, autotexts = ax.pie(level_dist, labels=level_dist.index, autopct='%1.1f%%',
                                    colors=colors, startangle=90, textprops={'fontsize': 12})

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

ax.set_title('Risk Level Distribution', fontsize=14, fontweight='bold')

# Add legend with counts
legend_labels = [f'{level}: {count} risks' for level, count in zip(level_dist.index, level_dist.values)]
ax.legend(legend_labels, loc='upper right', bbox_to_anchor=(1.15, 1), fontsize=10)

plt.tight_layout()
plt.show()

## Executive Summary Report

In [ ]:
# Generate formatted executive summary
print('\n' + '='*90)
print('EXECUTIVE SUMMARY - FINGUARD AI RISK ASSESSMENT')
print('='*90)

print(f'\nTotal Risks Identified: {len(df)}')
print(f'Average Risk Score: {df["risk_score"].mean():.2f}')

print('\n--- RISK LEVEL BREAKDOWN ---')
critical = len(df[df['risk_level'] == 'Critical'])
high = len(df[df['risk_level'] == 'High'])
medium = len(df[df['risk_level'] == 'Medium'])
low = len(df[df['risk_level'] == 'Low'])
print(f'  Critical: {critical} ({(critical/len(df))*100:.1f}%)')
print(f'  High:     {high} ({(high/len(df))*100:.1f}%)')
print(f'  Medium:   {medium} ({(medium/len(df))*100:.1f}%)')
print(f'  Low:      {low} ({(low/len(df))*100:.1f}%)')

print('\n--- TOP 5 PRIORITY RISKS ---')
top_5 = df_ranked.head(5)
for idx, row in top_5.iterrows():
    print(f'\n  {idx+1}. {row["id"]}: {row["description"][:70]}...')
    print(f'     Category: {row["category"]} | NIST Function: {row["nist_function"]}')
    print(f'     Impact: {row["impact"]}, Likelihood: {row["likelihood"]}, Score: {row["risk_score"]} ({row["risk_level"]})')

print('\n--- KEY FINDINGS ---')
print('  • FinGuard AI faces 4 Critical risks concentrated in technical model performance'
      ' and ethical bias. Immediate attention required.')
print('  • Legal risks (GDPR Articles 22, 13-14) represent significant regulatory exposure'
      ' from automated blocking and inadequate customer explanations.')
print('  • Operational risks are moderate (5 Medium). System redundancy and staff training'
      ' needed within 90 days.')
print('  • Primary mitigation focus: Bias detection (R002), Concept drift monitoring (R012),'
      ' and GDPR compliance (R007, R008).')

print('\n' + '='*90)

## Conclusion

This comprehensive risk analysis provides FinGuard AI with a clear prioritization framework for FraudShield system improvements. The 4 critical risks require executive escalation and immediate resource allocation, while high and medium risks should be addressed through a structured 90-day remediation plan. Regular monitoring and quarterly reassessment are essential given the dynamic nature of AI systems and evolving regulatory requirements.